## 6 — MRdeeP (state-level estimates, CES sample1)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
Outcomes extracted: `climate_problem`, `renewable_fuel`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts `climate_problem` and `renewable_fuel` estimates

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    BASE_DIR, OUTPUT_DIR, STATE_FIPS_TO_NAME,
    SURVEY_PATH,
    _recode_demographics,
)

TARGET_OUTCOMES = ['climate_problem', 'renewable_fuel']

### 1. Load and prepare data

In [ ]:
DATA_DIR = BASE_DIR / 'test_data' / 'processed'

# Survey: recode demographics, keep all binary outcome columns
raw = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
raw = _recode_demographics(raw)

# All outcome columns in the CES sample are already binary
OUTCOME_COLS = [c for c in raw.columns
                if c in ['climate_problem','regulate_carbon','renewable_fuel',
                         'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']]
DEMOG_VARS = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

survey = (
    raw[DEMOG_VARS + OUTCOME_COLS]
    .dropna()
    .copy()
)
survey['educ_category'] = survey['educ_category'].astype(str)

# Benchmark: county-level poststrat frame
ps_county = pd.read_csv(
    DATA_DIR / 'poststrat_county.csv',
    dtype={'state_fips': str, 'county_fips': str},
)
benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

### 2. Insert data into MRdeeP

In [ ]:
mod = MRdeeP(ensembles=3, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

### 3. Train CGAN ensemble
Default architecture: 4 × 256-neuron hidden layers, Wasserstein loss + gradient penalty.

In [ ]:
mod.fit(
    epochs        = 500,
    patience      = 50,
    batch_size    = 256,
    k             = 32,
    print_runtime = True,
)
print(mod)

### 4. Post-stratify → state-level estimates for all outcomes

In [ ]:
estimates = mod.post_stratify(levels='state_fips')
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

### 5. Extract target outcomes and save

In [ ]:
out_dir = OUTPUT_DIR / 'estimates'
out_dir.mkdir(parents=True, exist_ok=True)

for OUTCOME_VAR in TARGET_OUTCOMES:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)
    result = result.sort_values('estimate', ascending=False).reset_index(drop=True)

    out_path = out_dir / f'mrdeep_{OUTCOME_VAR}_state_estimates.csv'
    result[['state_fips', 'state_name', 'estimate']].to_csv(out_path, index=False)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(f'Top 5: {result.head(5)[["state_name","estimate"]].to_string(index=False)}')
    print(f'Saved → {out_path}')